# UNITPathSSL + DWT-FPN: Full Pipeline (Pretraining → Fine-tuning → Evaluation → Outputs)
Complete Kaggle notebook version of every script in `code/FPN`.

**Pipeline stages:**
1. **Stage A (optional):** Generate synthetic pseudo masks (`generate_pseudo_masks.py`)
2. **Stage B (optional):** UNITPathSSL self-supervised pretraining (`generator_g.py`, `pretrain_large.py`, `loss.py`)
3. **Stage C:** Supervised fine-tuning on MoNuSeg (`dataset.py`, `backbone.py`, `DWTFusion.py`, `ModifiedFPN.py`, `seg_head.py`, `train.py`)
4. **Stage D:** Test evaluation (`test.py`)
5. **Stage E:** Outputs — training curves (`plot_curves.py`), qualitative panels (`generate_output_mask.py`), unlabeled inference (`infer_unlabeled.py`), baseline comparison (`compare_baseline.py`)

**Before running:** Session options → Accelerator = **GPU**, Internet = **On**.

Stages A+B require an unlabeled H&E tissue dataset (e.g., Kather CRC-100K) and ~24h+ of GPU time for the full 65k iterations — they are **skippable**: set `RUN_PRETRAINING = False` to fine-tune directly from ImageNet weights (or from an uploaded UNITPathSSL checkpoint).

In [ ]:
# ============ 1. Imports & environment check ============
import os, glob, time, csv
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, jaccard_score,
                             precision_score, recall_score, roc_auc_score)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ============ 2. Configuration (replaces all argparse CLIs) ============
class Config:
    # ---- Pipeline switches ----
    RUN_PRETRAINING = False      # Stage A+B: needs tissue dataset + long GPU time
    RUN_UNLABELED_INFERENCE = False  # Stage E3: needs unlabeled tissue tiles

    # ---- MoNuSeg data (Kaggle paths) ----
    base = "/kaggle/input/datasets/tuanledinh/monuseg2018"
    train_img_dir  = f"{base}/kmms_training/kmms_training/images"
    train_mask_dir = f"{base}/kmms_training/kmms_training/masks"
    test_img_dir   = f"{base}/kmms_test/kmms_test/images"
    test_mask_dir  = f"{base}/kmms_test/kmms_test/masks"

    # ---- Pretraining (Stage A+B) ----
    pseudo_mask_dir = "/kaggle/working/data/pseudo_masks"
    num_pseudo_masks = 5000
    pseudo_mask_size = (256, 256)
    tissue_dir = "/kaggle/input/YOUR_TISSUE_DATASET"  # e.g. Kather CRC-100K tiles
    pretrain_size = (256, 256)
    pretrain_batch_size = 4
    pretrain_accum_steps = 6
    stage1_iters = 40000   # reduce drastically (e.g. 2000) for a Kaggle demo run
    stage2_iters = 25000
    pretrain_save_dir = "/kaggle/working/checkpoints/pretrain"

    # ---- Model ----
    backbone = "resnet50"
    pretrained = True            # ImageNet init for the backbone
    pretrained_weights = None    # path to UNITPathSSL ckpt; auto-filled if RUN_PRETRAINING
    num_classes = 1

    # ---- Fine-tuning (Stage C) — thesis hyperparameters, Kaggle-adapted ----
    lr = 5e-5
    weight_decay = 1e-4
    epochs = 30
    batch_size = 4               # was 1 on 4GB GPU; real batching on Kaggle T4/P100
    accumulation_steps = 1       # was 4; effective batch stays 4
    target_size = (512, 512)
    freeze_bn = True             # thesis invariant; try False with batch>=4

    # ---- Augmentation (rotate-and-crop version, no border artifacts) ----
    augment = True
    augment_prob = 0.6
    augment_factor = 3
    rotation_range = (-30, 30)
    scale_range = (0.85, 1.15)

    # ---- Runtime / outputs ----
    num_workers = 2
    save_dir = "/kaggle/working/checkpoints/final"
    panels_dir = "/kaggle/working/output/test_panels"
    unlabeled_out_dir = "/kaggle/working/data/unlabeled_predictions"
    threshold = 0.5
    title_tag = "UNITPathSSL + DWT-FPN"

cfg = Config()
for d in (cfg.save_dir, cfg.panels_dir):
    os.makedirs(d, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert os.path.isdir(cfg.train_img_dir), f"Image dir not found: {cfg.train_img_dir}"
print("Device:", device, "| Train images:", len(os.listdir(cfg.train_img_dir)))

## Stage A — Synthetic Pseudo Masks (`generate_pseudo_masks.py`)
Creates binary polygon masks that mimic nuclei layouts: 2–4 random "glandular" cluster centers, ~120 jittered polygons per mask (60% clustered near glands, 40% scattered). These form **Domain X** for the unpaired translation pretraining — no real annotations needed.

In [ ]:
# ============ 3. Pseudo mask generator ============
def generate_polygon_mask(size=(512, 512), num_nuclei=100, min_radius=4, max_radius=9,
                          num_vertices=16, distributiveness=0.33):
    """distributiveness in [0, 1]:
    0.0 -> tight clumps, nuclei may touch/overlap, strong clustering
    1.0 -> evenly spread, well-separated nuclei
    """
    mask = np.zeros(size, dtype=np.uint8)
    h, w = size
    d = float(np.clip(distributiveness, 0.0, 1.0))

    # Random glandular clusters (centers may sit near the edges too)
    num_glands = np.random.randint(2, 5)
    gland_centers = np.column_stack([
        np.random.randint(0, w, num_glands),
        np.random.randint(0, h, num_glands),
    ])

    placed_centers = []
    cluster_prob = 0.7 - 0.4 * d            # low d -> mostly clustered
    spread_lo, spread_hi = 10 + 30 * d, 40 + 90 * d  # low d -> tight clusters

    actual_nuclei = num_nuclei + np.random.randint(-15, 16)  # vary count +-15

    for _ in range(actual_nuclei):
        # random per-nucleus separation: gaps and closeness both occur
        base_sep = max_radius * (0.4 + 1.6 * d)          # 0 -> overlap allowed, 1 -> wide gaps
        min_sep = base_sep * np.random.uniform(0.5, 1.5)  # random gaps / random closeness

        cx, cy = None, None
        for _attempt in range(10):
            if np.random.rand() < cluster_prob and len(gland_centers) > 0:
                g_center = gland_centers[np.random.randint(0, num_glands)]
                angle = np.random.uniform(0, 2 * np.pi)
                r_dist = np.random.uniform(spread_lo, spread_hi)
                tx = int(np.clip(g_center[0] + r_dist * np.cos(angle), 0, w - 1))
                ty = int(np.clip(g_center[1] + r_dist * np.sin(angle), 0, h - 1))
            else:
                tx = np.random.randint(0, w)
                ty = np.random.randint(0, h)

            if all((tx - px) ** 2 + (ty - py) ** 2 >= min_sep ** 2 for px, py in placed_centers):
                cx, cy = tx, ty
                break
        if cx is None:
            continue  # couldn't find a spot respecting this nucleus' separation; skip
        placed_centers.append((cx, cy))

        # Jittered vertices -> convex hull (nuclei are convex; clipped at borders -> partial nuclei at edges)
        angles = np.sort(np.random.uniform(0, 2 * np.pi, num_vertices))
        radii = np.random.uniform(min_radius, max_radius, num_vertices)
        pts = [[int(cx + r * np.cos(a)), int(cy + r * np.sin(a))] for a, r in zip(angles, radii)]
        hull = cv2.convexHull(np.array(pts, dtype=np.int32))
        cv2.fillPoly(mask, [hull], 255)
    return mask


# Preview 3 pseudo masks (always safe to run)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax in axes:
    ax.imshow(generate_polygon_mask(size=cfg.pseudo_mask_size), cmap="gray")
    ax.axis("off")
plt.suptitle("Synthetic pseudo masks (Domain X)"); plt.show()

if cfg.RUN_PRETRAINING:
    os.makedirs(cfg.pseudo_mask_dir, exist_ok=True)
    print(f"Generating {cfg.num_pseudo_masks} masks in {cfg.pseudo_mask_dir}...")
    for idx in range(cfg.num_pseudo_masks):
        m = generate_polygon_mask(size=cfg.pseudo_mask_size)
        cv2.imwrite(os.path.join(cfg.pseudo_mask_dir, f"pseudo_mask_{idx:05d}.png"), m)
    print("Pseudo mask generation complete.")

In [ ]:
# ============ 3b. Sanity tests for generate_polygon_mask ============
def test_generate_polygon_mask():
    # shape & dtype
    for size in [(512, 512), (256, 256)]:
        m = generate_polygon_mask(size=size)
        assert m.shape == size, f"expected shape {size}, got {m.shape}"
        assert m.dtype == np.uint8, f"expected uint8, got {m.dtype}"
    print("PASS shape & dtype")

    # strictly binary (only 0 and 255)
    m = generate_polygon_mask()
    vals = set(np.unique(m).tolist())
    assert vals <= {0, 255}, f"non-binary values found: {vals}"
    print("PASS binary values (only 0 and 255)")

    # foreground coverage: never empty, never mostly white
    ratios = [(generate_polygon_mask() == 255).mean() for _ in range(5)]
    mean_ratio = float(np.mean(ratios))
    assert 0.01 < mean_ratio < 0.5, f"suspicious foreground ratio {mean_ratio:.3f}"
    print(f"PASS foreground coverage (mean {mean_ratio:.1%})")

    # connected components: close nuclei may merge, so allow a wide range
    m = generate_polygon_mask()
    num_labels, _, stats, _ = cv2.connectedComponentsWithStats(m)
    n_blobs = num_labels - 1  # minus background
    assert 20 <= n_blobs <= 115, f"unexpected component count {n_blobs}"
    areas = stats[1:, cv2.CC_STAT_AREA]
    assert areas.min() >= 3, f"component too small: {areas.min()} px"
    print(f"PASS components: {n_blobs} blobs, area range {areas.min()}-{areas.max()} px")

    # edge behavior: nuclei are allowed to touch/overlap the image border now,
    # so just check the distributiveness extremes produce different separations
    clumped = generate_polygon_mask(distributiveness=0.0)
    spread = generate_polygon_mask(distributiveness=1.0)
    n_c = cv2.connectedComponentsWithStats(clumped)[0] - 1
    n_s = cv2.connectedComponentsWithStats(spread)[0] - 1
    assert n_c < n_s, f"expected clumped ({n_c}) to merge more than spread ({n_s})"
    print(f"PASS distributiveness (clumped {n_c} blobs < spread {n_s} blobs)")

    # randomness: two consecutive calls must differ
    assert not np.array_equal(generate_polygon_mask(), generate_polygon_mask()), \
        "two calls produced identical masks"
    print("PASS randomness (consecutive masks differ)")

    print("\nAll pseudo-mask tests passed.")

test_generate_polygon_mask()

## Model Components (`loss.py`, `DWTFusion.py`, `ModifiedFPN.py`, `backbone.py`, `seg_head.py`, `generator_g.py`)
Everything the pipeline shares, defined once:
- **Losses:** BCE+Dice for supervised fine-tuning; `UNITPathSSLLoss` (adversarial + cycle + ISG) for pretraining.
- **DWT-FPN stack:** `DWTEntropyFusion` → `ModifiedFPN` → `SegModel` (this *is* the Segmentor **S** during pretraining AND the downstream model).
- **`CoModulatedGenerator` G:** pseudo mask + noise → synthetic H&E image (StyleGAN2-style weight demodulation).

In [ ]:
# ============ 4. Losses (loss.py) ============
bce_loss = nn.BCEWithLogitsLoss()

def dice_loss(pred, target, smooth=1.0):
    pred = torch.sigmoid(pred).view(-1)
    target = target.view(-1)
    intersection = (pred * target).sum()
    return 1.0 - ((2.0 * intersection + smooth) / (pred.sum() + target.sum() + smooth))

def total_loss(pred, target):
    """Downstream Supervised Fine-Tuning Loss."""
    return bce_loss(pred, target) + dice_loss(pred, target)


class UNITPathSSLLoss(nn.Module):
    """L = L_GAN(G,DG) + l1*L_GAN(S,DS) + l2*L_cyc(G,S) + l3*L_seg(S)"""
    def __init__(self, lambda1=2.0, lambda2=10.0, lambda3=2.0):
        super().__init__()
        self.lambda1, self.lambda2, self.lambda3 = lambda1, lambda2, lambda3
        self.l1_loss = nn.L1Loss()

    def forward(self, d_s_pred_fake, x_rec, real_mask_x, seg_loss_dict=None):
        loss_adv_s = 0.5 * torch.mean((d_s_pred_fake - 1.0) ** 2)   # LSGAN on Segmentor S
        loss_cyc = self.l1_loss(x_rec, real_mask_x)                 # L1 cycle consistency
        loss_isg = torch.tensor(0.0, device=x_rec.device)
        if seg_loss_dict is not None:
            loss_isg = sum(loss for loss in seg_loss_dict.values())
        total = (self.lambda1 * loss_adv_s) + (self.lambda2 * loss_cyc) + (self.lambda3 * loss_isg)
        return total, {"loss_adv_s": loss_adv_s.item(), "loss_cyc": loss_cyc.item(), "loss_isg": loss_isg.item()}

In [ ]:
# ============ 5. DWT Entropy Fusion (DWTFusion.py) ============
class DWTEntropyFusion(nn.Module):
    """Entropy-guided spatial attention fusion for Wavelet high-frequency components."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv_reduce = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.attn_conv = nn.Sequential(
            nn.Conv2d(in_channels * 3, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 3, kernel_size=1))

    def local_entropy(self, x):
        p = torch.abs(x)
        p = F.avg_pool2d(p, kernel_size=3, stride=1, padding=1)
        p = p / (p.sum(dim=(2, 3), keepdim=True) + 1e-8)
        ent = -p * torch.log(p + 1e-8)
        return torch.mean(ent, dim=1, keepdim=True)

    def forward(self, LH, HL, HH):
        LH = (LH - LH.mean(dim=(2, 3), keepdim=True)) / (LH.std(dim=(2, 3), keepdim=True) + 1e-6)
        HL = (HL - HL.mean(dim=(2, 3), keepdim=True)) / (HL.std(dim=(2, 3), keepdim=True) + 1e-6)
        HH = (HH - HH.mean(dim=(2, 3), keepdim=True)) / (HH.std(dim=(2, 3), keepdim=True) + 1e-6)

        entropy_weights = torch.cat([self.local_entropy(LH), self.local_entropy(HL),
                                     self.local_entropy(HH)], dim=1)
        attn = self.attn_conv(torch.cat([LH, HL, HH], dim=1))
        weights = F.softmax(entropy_weights + attn, dim=1)
        F_out = weights[:, 0:1] * LH + weights[:, 1:2] * HL + weights[:, 2:3] * HH
        return self.conv_reduce(F_out)

In [ ]:
# ============ 6. Modified FPN (ModifiedFPN.py) ============
class ModifiedFPN(nn.Module):
    """FPN with GroupNorm (32 groups), DWT Entropy Fusion, and P6 level."""
    def __init__(self, in_channels=(256, 512, 1024, 2048), out_channels=256, num_groups=32):
        super().__init__()
        if len(in_channels) != 4:
            raise ValueError("ModifiedFPN requires exactly 4 backbone feature channels")

        def lat(c_in):
            return nn.Sequential(nn.Conv2d(c_in, out_channels, kernel_size=1),
                                 nn.GroupNorm(num_groups, out_channels))
        self.lat2, self.lat3 = lat(in_channels[0]), lat(in_channels[1])
        self.lat4, self.lat5 = lat(in_channels[2]), lat(in_channels[3])

        def smooth():
            return nn.Sequential(
                nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
                nn.GroupNorm(num_groups, out_channels),
                nn.LeakyReLU(0.2, inplace=True))
        self.smooth2, self.smooth3 = smooth(), smooth()
        self.smooth4, self.smooth5 = smooth(), smooth()

        self.p6 = nn.MaxPool2d(kernel_size=1, stride=2, padding=0)

        self.dwt2_fuse = DWTEntropyFusion(in_channels[0], out_channels)
        self.dwt3_fuse = DWTEntropyFusion(in_channels[1], out_channels)
        self.dwt4_fuse = DWTEntropyFusion(in_channels[2], out_channels)

        self.register_buffer("ll", torch.tensor([[0.5, 0.5], [0.5, 0.5]]))
        self.register_buffer("lh", torch.tensor([[-0.5, -0.5], [0.5, 0.5]]))
        self.register_buffer("hl", torch.tensor([[-0.5, 0.5], [-0.5, 0.5]]))
        self.register_buffer("hh", torch.tensor([[0.5, -0.5], [-0.5, 0.5]]))

    def dwt(self, x):
        B, C, H, W = x.shape
        if H % 2 != 0 or W % 2 != 0:
            x = F.pad(x, (0, W % 2, 0, H % 2))
        filters = torch.stack([self.ll, self.lh, self.hl, self.hh], dim=0).unsqueeze(1)
        filters = filters.repeat(C, 1, 1, 1)
        out = F.conv2d(x, filters, stride=2, groups=C)
        out = out.view(B, C, 4, out.shape[2], out.shape[3])
        return out[:, :, 1], out[:, :, 2], out[:, :, 3]

    def forward(self, C2, C3, C4, C5):
        P5_lat = self.lat5(C5)
        P4_lat = self.lat4(C4)
        P3_lat = self.lat3(C3)
        P2_lat = self.lat2(C2)

        P4 = P4_lat + F.interpolate(P5_lat, size=P4_lat.shape[2:], mode='nearest')
        P3 = P3_lat + F.interpolate(P4, size=P3_lat.shape[2:], mode='nearest')
        P2 = P2_lat + F.interpolate(P3, size=P2_lat.shape[2:], mode='nearest')

        LH2, HL2, HH2 = self.dwt(C2)
        LH3, HL3, HH3 = self.dwt(C3)
        LH4, HL4, HH4 = self.dwt(C4)

        F2 = F.interpolate(self.dwt2_fuse(LH2, HL2, HH2), size=P2.shape[2:], mode='bilinear', align_corners=False)
        F3 = F.interpolate(self.dwt3_fuse(LH3, HL3, HH3), size=P3.shape[2:], mode='bilinear', align_corners=False)
        F4 = F.interpolate(self.dwt4_fuse(LH4, HL4, HH4), size=P4.shape[2:], mode='bilinear', align_corners=False)

        P2 = self.smooth2(P2 + F2)
        P3 = self.smooth3(P3 + F3)
        P4 = self.smooth4(P4 + F4)
        P5 = self.smooth5(P5_lat)
        P6 = self.p6(P5)
        return P2, P3, P4, P5, P6

In [ ]:
# ============ 7. Backbones (backbone.py) ============
from torchvision.models import (
    resnet18, resnet34, resnet50,
    ResNet18_Weights, ResNet34_Weights, ResNet50_Weights,
    mobilenet_v2, MobileNet_V2_Weights,
    efficientnet_b0, EfficientNet_B0_Weights,
    densenet121, DenseNet121_Weights,
)


def _get_resnet_channels(resnet):
    def _layer_channels(layer):
        block = layer[-1]
        return block.bn3.num_features if hasattr(block, 'bn3') else block.bn2.num_features
    return [_layer_channels(resnet.layer1), _layer_channels(resnet.layer2),
            _layer_channels(resnet.layer3), _layer_channels(resnet.layer4)]


class ResNetBackbone(nn.Module):
    def __init__(self, variant="resnet50", pretrained=False):
        super().__init__()
        weights_map = {"resnet18": ResNet18_Weights.IMAGENET1K_V1,
                       "resnet34": ResNet34_Weights.IMAGENET1K_V1,
                       "resnet50": ResNet50_Weights.IMAGENET1K_V1}
        if variant not in weights_map:
            raise ValueError(f"Unsupported ResNet variant: {variant}")
        resnet_fn = {"resnet18": resnet18, "resnet34": resnet34, "resnet50": resnet50}[variant]
        resnet = resnet_fn(weights=weights_map[variant] if pretrained else None)

        self.stage0 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.stage1, self.stage2 = resnet.layer1, resnet.layer2
        self.stage3, self.stage4 = resnet.layer3, resnet.layer4
        self.out_channels = _get_resnet_channels(resnet)

    def forward(self, x):
        x = self.stage0(x)
        C2 = self.stage1(x); C3 = self.stage2(C2)
        C4 = self.stage3(C3); C5 = self.stage4(C4)
        return C2, C3, C4, C5


class MobileNetV2Backbone(nn.Module):
    def __init__(self, pretrained=False):
        super().__init__()
        model = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None)
        self.stage0 = nn.Sequential(model.features[0], model.features[1], model.features[2])
        self.stage1 = nn.Sequential(*model.features[3:7])
        self.stage2 = nn.Sequential(*model.features[7:14])
        self.stage3 = nn.Sequential(*model.features[14:])
        self.out_channels = [24, 32, 96, 1280]

    def forward(self, x):
        C2 = self.stage0(x); C3 = self.stage1(C2)
        C4 = self.stage2(C3); C5 = self.stage3(C4)
        return C2, C3, C4, C5


class EfficientNetBackbone(nn.Module):
    def __init__(self, pretrained=False):
        super().__init__()
        model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
        self.stage0 = nn.Sequential(model.features[0], model.features[1], model.features[2])
        self.stage1 = nn.Sequential(model.features[3])
        self.stage2 = nn.Sequential(model.features[4])
        self.stage3 = nn.Sequential(*model.features[5:])
        self.out_channels = [24, 40, 80, 1280]

    def forward(self, x):
        C2 = self.stage0(x); C3 = self.stage1(C2)
        C4 = self.stage2(C3); C5 = self.stage3(C4)
        return C2, C3, C4, C5


class DenseNetBackbone(nn.Module):
    def __init__(self, pretrained=False):
        super().__init__()
        model = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None)
        self.stage0 = nn.Sequential(*model.features[:5])
        self.stage1 = nn.Sequential(model.features[5])
        self.stage2 = nn.Sequential(model.features[6], model.features[7])
        self.stage3 = nn.Sequential(*model.features[8:])
        self.out_channels = [256, 128, 256, 1024]

    def forward(self, x):
        C2 = self.stage0(x); C3 = self.stage1(C2)
        C4 = self.stage2(C3); C5 = self.stage3(C4)
        return C2, C3, C4, C5


def build_backbone(backbone_name="resnet50", pretrained=False):
    backbone_name = backbone_name.lower()
    if backbone_name == "resnet20":
        backbone_name = "resnet18"
    if backbone_name in {"resnet18", "resnet34", "resnet50"}:
        return ResNetBackbone(variant=backbone_name, pretrained=pretrained)
    if backbone_name == "mobilenet_v2":
        return MobileNetV2Backbone(pretrained=pretrained)
    if backbone_name == "efficientnet_b0":
        return EfficientNetBackbone(pretrained=pretrained)
    if backbone_name == "densenet121":
        return DenseNetBackbone(pretrained=pretrained)
    raise ValueError(f"Unsupported backbone: {backbone_name}")

In [ ]:
# ============ 8. Segmentation model S (seg_head.py) ============
class SegModel(nn.Module):
    """Backbone + DWT-FPN + multi-scale summation decoder. Used as Segmentor S in
    pretraining AND as the downstream fine-tuned model."""
    def __init__(self, backbone_name="resnet50", pretrained=True, fpn_out_channels=256, num_classes=1):
        super().__init__()
        self.backbone_name = backbone_name
        self.backbone = build_backbone(backbone_name, pretrained=pretrained)
        self.fpn = ModifiedFPN(in_channels=self.backbone.out_channels, out_channels=fpn_out_channels)

        self.decoder_conv = nn.Sequential(
            nn.Conv2d(fpn_out_channels, fpn_out_channels // 2, kernel_size=3, padding=1),
            nn.InstanceNorm2d(fpn_out_channels // 2, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(fpn_out_channels // 2, fpn_out_channels // 4, kernel_size=3, padding=1),
            nn.InstanceNorm2d(fpn_out_channels // 4, affine=True),
            nn.LeakyReLU(0.2, inplace=True))
        self.final_upsample = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(fpn_out_channels // 4, fpn_out_channels // 4, kernel_size=3, padding=1),
            nn.InstanceNorm2d(fpn_out_channels // 4, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(fpn_out_channels // 4, num_classes, kernel_size=3, padding=1))

    def extract_features(self, x):
        C2, C3, C4, C5 = self.backbone(x)
        return self.fpn(C2, C3, C4, C5)

    def forward(self, x):
        P2, P3, P4, P5, P6 = self.extract_features(x)
        target_size = P2.shape[2:]
        u3 = F.interpolate(P3, size=target_size, mode="bilinear", align_corners=False)
        u4 = F.interpolate(P4, size=target_size, mode="bilinear", align_corners=False)
        u5 = F.interpolate(P5, size=target_size, mode="bilinear", align_corners=False)
        u6 = F.interpolate(P6, size=target_size, mode="bilinear", align_corners=False)
        fused = P2 + u3 + u4 + u5 + u6
        return self.final_upsample(self.decoder_conv(fused))

In [ ]:
# ============ 9. Co-modulated Generator G (generator_g.py) ============
class ModulatedConv2d(nn.Module):
    """Weight-demodulation conv (StyleGAN2/CoModGAN): per-sample style modulates weights."""
    def __init__(self, in_channels, out_channels, kernel_size=3, style_dim=512):
        super().__init__()
        self.in_channels, self.out_channels = in_channels, out_channels
        self.kernel_size, self.padding = kernel_size, kernel_size // 2
        self.weight = nn.Parameter(torch.randn(out_channels, in_channels, kernel_size, kernel_size))
        self.affine = nn.Linear(style_dim, in_channels)

    def forward(self, x, style):
        B, C, H, W = x.shape
        s = self.affine(style).view(B, 1, self.in_channels, 1, 1) + 1.0
        w = self.weight.unsqueeze(0) * s
        d = torch.rsqrt(w.pow(2).sum(dim=[2, 3, 4], keepdim=True) + 1e-8)
        w_demod = (w * d).view(B * self.out_channels, self.in_channels,
                               self.kernel_size, self.kernel_size)
        x = x.view(1, B * C, H, W)
        out = F.conv2d(x, w_demod, padding=self.padding, groups=B)
        return out.view(B, self.out_channels, H, W)


class CoModulatedGenerator(nn.Module):
    """G: pseudo mask x + noise z -> synthetic H&E image y_hat in [-1, 1]."""
    def __init__(self, in_channels=1, out_channels=3, z_dim=512, style_dim=512):
        super().__init__()
        self.z_dim, self.style_dim = z_dim, style_dim

        self.mapping_net = nn.Sequential(
            nn.Linear(z_dim, style_dim), nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(style_dim, style_dim), nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(style_dim, style_dim), nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(style_dim, style_dim), nn.LeakyReLU(0.2, inplace=True))

        self.enc1 = nn.Conv2d(in_channels, 64, kernel_size=3, padding=1)
        self.down1 = nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1)
        self.down2 = nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1)
        self.down3 = nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1)
        self.down4 = nn.Conv2d(512, 512, kernel_size=4, stride=2, padding=1)
        self.down5 = nn.Conv2d(512, 512, kernel_size=4, stride=2, padding=1)
        self.down6 = nn.Conv2d(512, 512, kernel_size=4, stride=2, padding=1)

        self.enc_fc = nn.Sequential(nn.Flatten(), nn.Dropout(0.5),
                                    nn.Linear(512 * 4 * 4, style_dim),
                                    nn.LeakyReLU(0.2, inplace=True))
        self.init_proj = nn.Linear(style_dim, 512 * 4 * 4)
        self.fusion = nn.Sequential(nn.Linear(style_dim * 2, style_dim),
                                    nn.LeakyReLU(0.2, inplace=True))

        self.mod1 = ModulatedConv2d(512, 512, 3, style_dim)
        self.up1 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.skip1 = nn.Conv2d(512, 512, kernel_size=1)
        self.mod2 = ModulatedConv2d(512, 512, 3, style_dim)
        self.up2 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.skip2 = nn.Conv2d(512, 512, kernel_size=1)
        self.mod3 = ModulatedConv2d(512, 256, 3, style_dim)
        self.up3 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.skip3 = nn.Conv2d(512, 256, kernel_size=1)
        self.mod4 = ModulatedConv2d(256, 128, 3, style_dim)
        self.up4 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.skip4 = nn.Conv2d(256, 128, kernel_size=1)
        self.mod5 = ModulatedConv2d(128, 64, 3, style_dim)
        self.up5 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.skip5 = nn.Conv2d(128, 64, kernel_size=1)
        self.mod6 = ModulatedConv2d(64, 32, 3, style_dim)
        self.up6 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.to_rgb = nn.Conv2d(32, out_channels, kernel_size=1)

    def forward(self, mask_x, z=None):
        B = mask_x.shape[0]
        if z is None:
            z = torch.randn(B, self.z_dim, device=mask_x.device)

        e1 = F.leaky_relu(self.enc1(mask_x), 0.2)
        e2 = F.leaky_relu(self.down1(e1), 0.2)
        e3 = F.leaky_relu(self.down2(e2), 0.2)
        e4 = F.leaky_relu(self.down3(e3), 0.2)
        e5 = F.leaky_relu(self.down4(e4), 0.2)
        e6 = F.leaky_relu(self.down5(e5), 0.2)
        e7 = F.leaky_relu(self.down6(e6), 0.2)

        E_x = self.enc_fc(e7)
        w = self.mapping_net(z)
        style_s = self.fusion(torch.cat([E_x, w], dim=1))

        h = self.init_proj(E_x).view(B, 512, 4, 4)
        h = F.leaky_relu(self.mod1(h, style_s), 0.2)
        h = self.up1(h) + self.skip1(e6)
        h = F.leaky_relu(self.mod2(h, style_s), 0.2)
        h = self.up2(h) + self.skip2(e5)
        h = F.leaky_relu(self.mod3(h, style_s), 0.2)
        h = self.up3(h) + self.skip3(e4)
        h = F.leaky_relu(self.mod4(h, style_s), 0.2)
        h = self.up4(h) + self.skip4(e3)
        h = F.leaky_relu(self.mod5(h, style_s), 0.2)
        h = self.up5(h) + self.skip5(e2)
        h = F.leaky_relu(self.mod6(h, style_s), 0.2)
        h = self.up6(h)
        return torch.tanh(self.to_rgb(h))

## Stage B — UNITPathSSL Pretraining (`pretrain_large.py`) — optional
Unpaired translation between pseudo masks (X) and real tissue patches (Y):
- **G** (CoModulatedGenerator): mask → fake tissue; **S** (SegModel): tissue → mask.
- **Forward cycle** x → G(x) → S(G(x)) ≈ x enforced with L1; PatchGAN discriminators D_G/D_S with LSGAN loss.
- Two stages: Stage 1 (40k iters @ 4e-4), then S and D_S reinitialized for Stage 2 (25k iters @ 1e-4). Only **S's weights** are kept — they become the fine-tuning initialization.

⚠️ Needs `cfg.tissue_dir` pointing at unlabeled H&E patches, and the full run exceeds a single Kaggle session — reduce `stage1_iters`/`stage2_iters` for demos, or skip and use ImageNet init.

In [ ]:
# ============ 10. Pretraining dataset + discriminator + stage runner ============
class LargeUnpairedDataset(Dataset):
    def __init__(self, mask_dir, tissue_dir, target_size=(256, 256)):
        super().__init__()
        valid_exts = (".png", ".tif", ".tiff", ".jpg", ".jpeg")
        self.mask_files = sorted([f for f in glob.glob(os.path.join(mask_dir, "**", "*.*"), recursive=True)
                                  if f.lower().endswith(valid_exts)])
        self.tissue_files = sorted([f for f in glob.glob(os.path.join(tissue_dir, "**", "*.*"), recursive=True)
                                    if f.lower().endswith(valid_exts)])
        self.target_size = tuple(target_size)
        if len(self.tissue_files) == 0 or len(self.mask_files) == 0:
            raise ValueError(f"Found {len(self.tissue_files)} tissues / {len(self.mask_files)} masks.")
        print(f"Dataset ready: {len(self.tissue_files)} tissue images, {len(self.mask_files)} pseudo masks.")

    def __len__(self):
        return max(len(self.mask_files), len(self.tissue_files))

    def __getitem__(self, idx):
        m = cv2.imread(self.mask_files[idx % len(self.mask_files)], 0)
        m = cv2.resize(m, self.target_size, interpolation=cv2.INTER_NEAREST)
        mask_t = torch.from_numpy((m > 0).astype("float32")).unsqueeze(0)

        t = cv2.imread(self.tissue_files[idx % len(self.tissue_files)])
        t = cv2.cvtColor(t, cv2.COLOR_BGR2RGB)
        t = cv2.resize(t, self.target_size, interpolation=cv2.INTER_LINEAR)
        tissue_t = (torch.from_numpy(t).permute(2, 0, 1).float() / 127.5) - 1.0  # [-1, 1]
        return mask_t, tissue_t


class PatchDiscriminator(nn.Module):
    def __init__(self, in_channels=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.GroupNorm(16, 128), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.GroupNorm(32, 256), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 1, kernel_size=4, stride=1, padding=1))

    def forward(self, x):
        return self.net(x)


def run_stage(stage_num, max_iters, lr, gen_g, gen_s, disc_g, disc_s, loader,
              device, accum_steps, save_dir, backbone):
    opt_g = optim.Adam(list(gen_g.parameters()) + list(gen_s.parameters()), lr=lr, betas=(0.0, 0.99), eps=1e-8)
    opt_d = optim.Adam(list(disc_g.parameters()) + list(disc_s.parameters()), lr=lr, betas=(0.0, 0.99), eps=1e-8)
    criterion = UNITPathSSLLoss(lambda1=2.0, lambda2=10.0, lambda3=0.0)
    mse = nn.MSELoss()

    iter_cnt = 0
    data_iter = iter(loader)
    print(f"\n===== Pretraining Stage {stage_num}: {max_iters} iters @ LR={lr} =====")

    while iter_cnt < max_iters:
        try:
            real_x, real_y = next(data_iter)
        except StopIteration:
            data_iter = iter(loader)
            real_x, real_y = next(data_iter)
        real_x, real_y = real_x.to(device), real_y.to(device)

        # 1. Discriminator step
        opt_d.zero_grad()
        with torch.no_grad():
            fake_y = gen_g(real_x)
            fake_x = torch.sigmoid(gen_s((real_y + 1.0) / 2.0))
        d_g_loss = 0.5 * (mse(disc_g(real_y), torch.ones_like(disc_g(real_y))) +
                          mse(disc_g(fake_y.detach()), torch.zeros_like(disc_g(fake_y))))
        d_s_loss = 0.5 * (mse(disc_s(real_x), torch.ones_like(disc_s(real_x))) +
                          mse(disc_s(fake_x.detach()), torch.zeros_like(disc_s(fake_x))))
        loss_d = (d_g_loss + d_s_loss) / accum_steps
        loss_d.backward()

        # 2. Generator step (forward cycle x -> G(x) -> S(G(x)) ~ x)
        opt_g.zero_grad()
        fake_y = gen_g(real_x)
        loss_gan_g = mse(disc_g(fake_y), torch.ones_like(disc_g(fake_y)))
        rec_x = gen_s((fake_y + 1.0) / 2.0)
        d_s_fake = disc_s(torch.sigmoid(rec_x))
        loss_unit, _ = criterion(d_s_fake, torch.sigmoid(rec_x), real_x)
        loss_g_total = (loss_gan_g + loss_unit) / accum_steps
        loss_g_total.backward()

        if (iter_cnt + 1) % accum_steps == 0:
            torch.nn.utils.clip_grad_norm_(gen_s.parameters(), 1.0)
            opt_d.step()
            opt_g.step()

        iter_cnt += 1
        if iter_cnt % 500 == 0 or iter_cnt == max_iters:
            print(f"Stage {stage_num} | Iter [{iter_cnt:05d}/{max_iters:05d}] | "
                  f"Loss Gen: {loss_g_total.item() * accum_steps:.4f} | Loss D: {loss_d.item() * accum_steps:.4f}")

    ckpt_path = os.path.join(save_dir, f"{backbone}_stage{stage_num}_pretrained.pth")
    torch.save(gen_s.state_dict(), ckpt_path)
    print(f"--> Saved Stage {stage_num} checkpoint: {ckpt_path}")
    return ckpt_path

In [ ]:
# ============ 11. Run pretraining (skipped unless RUN_PRETRAINING) ============
if cfg.RUN_PRETRAINING:
    os.makedirs(cfg.pretrain_save_dir, exist_ok=True)
    pre_ds = LargeUnpairedDataset(cfg.pseudo_mask_dir, cfg.tissue_dir, cfg.pretrain_size)
    pre_loader = DataLoader(pre_ds, batch_size=cfg.pretrain_batch_size, shuffle=True,
                            num_workers=cfg.num_workers, pin_memory=True)

    gen_s = SegModel(backbone_name=cfg.backbone, pretrained=True).to(device)
    gen_g = CoModulatedGenerator(in_channels=1, out_channels=3).to(device)
    disc_g = PatchDiscriminator(in_channels=3).to(device)
    disc_s = PatchDiscriminator(in_channels=1).to(device)

    run_stage(1, cfg.stage1_iters, 4.0e-4, gen_g, gen_s, disc_g, disc_s, pre_loader,
              device, cfg.pretrain_accum_steps, cfg.pretrain_save_dir, cfg.backbone)

    # Stage 2: re-initialize S and D_S, keep G
    gen_s = SegModel(backbone_name=cfg.backbone, pretrained=True).to(device)
    disc_s = PatchDiscriminator(in_channels=1).to(device)
    stage2_ckpt = run_stage(2, cfg.stage2_iters, 1.0e-4, gen_g, gen_s, disc_g, disc_s, pre_loader,
                            device, cfg.pretrain_accum_steps, cfg.pretrain_save_dir, cfg.backbone)

    cfg.pretrained_weights = stage2_ckpt   # feed into fine-tuning
    del gen_g, gen_s, disc_g, disc_s, pre_loader, pre_ds
    torch.cuda.empty_cache()
else:
    print("Pretraining skipped (RUN_PRETRAINING=False).")
    print("Fine-tuning will use",
          f"UNITPathSSL checkpoint: {cfg.pretrained_weights}" if cfg.pretrained_weights
          else "ImageNet initialization.")

## Stage C — MoNuSeg Dataset & Fine-tuning (`dataset.py`, `train.py`)
Includes all recent fixes:
- **Mask lookup** tries `{base}_bin_mask.png` → `{base}.png` → `{base} .png` (trailing-space names in the Kaggle dataset), with `os.path.exists` checks so OpenCV prints no warnings.
- **Rotate-and-crop augmentation:** the random scale is clamped to `cosθ+sinθ` so the rotated image always covers the full canvas — no mirror seams or black border wedges, and image/mask stay perfectly aligned.

In [ ]:
# ============ 12. MoNuSeg dataset (dataset.py, with recent fixes) ============
def list_image_files(img_dir, extensions=None):
    if extensions is None:
        extensions = (".png", ".tif", ".tiff", ".jpg", ".jpeg")
    return sorted([f for f in os.listdir(img_dir) if f.lower().endswith(extensions)])


def split_image_list(image_files, test_size=0.1, val_size=0.1, random_state=None):
    if val_size < 0 or test_size < 0 or val_size + test_size >= 1.0:
        raise ValueError("val_size and test_size must be non-negative and sum to less than 1.0")
    train_val, test_files = train_test_split(image_files, test_size=test_size,
                                             random_state=random_state, shuffle=True)
    if val_size == 0.0:
        return train_val, [], test_files
    val_fraction = val_size / (1.0 - test_size)
    train_files, val_files = train_test_split(train_val, test_size=val_fraction,
                                              random_state=random_state, shuffle=True)
    return train_files, val_files, test_files


class MoNuSegDataset(Dataset):
    def __init__(self, img_dir, mask_dir, target_size=(512, 512), augment=False,
                 augment_factor=1, augment_prob=0.5, rotation_range=(-15, 15),
                 scale_range=(0.9, 1.1), image_list=None):
        self.img_dir, self.mask_dir = img_dir, mask_dir
        self.target_size = tuple(target_size)
        self.augment = augment
        self.augment_factor = augment_factor
        self.augment_probability = augment_prob
        self.rotation_range = rotation_range
        self.scale_range = scale_range

        valid_extensions = (".png", ".tif", ".tiff", ".jpg", ".jpeg")
        if image_list is None:
            self.images = list_image_files(img_dir, extensions=valid_extensions)
        else:
            self.images = [f for f in image_list if f.lower().endswith(valid_extensions)]

        self.samples = []
        for img_name in self.images:
            self.samples.append({"img_name": img_name, "augment": False})
            if self.augment:
                for _ in range(self.augment_factor):
                    self.samples.append({"img_name": img_name, "augment": True})
        if self.augment:
            np.random.shuffle(self.samples)

    def __len__(self):
        return len(self.samples)

    def _augment_pair(self, img, mask):
        angle = np.random.uniform(self.rotation_range[0], self.rotation_range[1])
        scale = np.random.uniform(self.scale_range[0], self.scale_range[1])
        h, w = img.shape[:2]
        center = (w / 2.0, h / 2.0)

        # Rotate-and-crop: zoom so the rotated image always covers the canvas
        rad = np.deg2rad(abs(angle))
        cover_scale = np.cos(rad) + np.sin(rad)
        scale = max(scale, cover_scale)

        matrix = cv2.getRotationMatrix2D(center, angle, scale)
        img = cv2.warpAffine(img, matrix, (w, h), flags=cv2.INTER_LINEAR)
        mask = cv2.warpAffine(mask, matrix, (w, h), flags=cv2.INTER_NEAREST)
        return img, mask

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img_name = sample["img_name"]
        should_augment = sample["augment"]

        img_path = os.path.join(self.img_dir, img_name)
        img = cv2.imread(img_path)
        if img is None:
            raise FileNotFoundError(f"Could not read image: {img_path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        base_name = os.path.splitext(img_name)[0]
        mask_candidates = (f"{base_name}_bin_mask.png", f"{base_name}.png", f"{base_name} .png")
        mask = None
        for mask_name in mask_candidates:
            mask_path = os.path.join(self.mask_dir, mask_name)
            if os.path.exists(mask_path):
                mask = cv2.imread(mask_path, 0)
                if mask is not None:
                    break
        if mask is None:
            raise FileNotFoundError(f"Could not find mask for {img_name} at {self.mask_dir}")

        img = cv2.resize(img, self.target_size, interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, self.target_size, interpolation=cv2.INTER_NEAREST)

        if should_augment and (np.random.rand() < self.augment_probability):
            img, mask = self._augment_pair(img, mask)

        img = np.ascontiguousarray(img)
        mask = np.ascontiguousarray(mask)

        mask = (mask > 0).astype("float32")
        img_tensor = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        mask_tensor = torch.from_numpy(mask).unsqueeze(0)
        return img_tensor, mask_tensor

In [ ]:
# ============ 13. Augmentation demo: before vs. after ============
demo_ds = MoNuSegDataset(cfg.train_img_dir, cfg.train_mask_dir,
                         target_size=cfg.target_size, augment=False)

n_examples = 3
np.random.seed(7)  # remove for different transforms each run
fig, axes = plt.subplots(n_examples, 4, figsize=(16, 4 * n_examples))

for row in range(n_examples):
    img_name = demo_ds.images[row]
    img = cv2.imread(os.path.join(demo_ds.img_dir, img_name))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    base = os.path.splitext(img_name)[0]
    mask = None
    for cand in (f"{base}_bin_mask.png", f"{base}.png", f"{base} .png"):
        cand_path = os.path.join(demo_ds.mask_dir, cand)
        if os.path.exists(cand_path):
            mask = cv2.imread(cand_path, 0)
            if mask is not None:
                break
    img = cv2.resize(img, demo_ds.target_size, interpolation=cv2.INTER_LINEAR)
    mask = cv2.resize(mask, demo_ds.target_size, interpolation=cv2.INTER_NEAREST)

    aug_img, aug_mask = demo_ds._augment_pair(img.copy(), mask.copy())

    axes[row, 0].imshow(img);                   axes[row, 0].set_title(f"Original\n{img_name}", fontsize=9)
    axes[row, 1].imshow(mask, cmap="gray");     axes[row, 1].set_title("Original Mask")
    axes[row, 2].imshow(aug_img);               axes[row, 2].set_title("Augmented (rotate + crop)")
    axes[row, 3].imshow(aug_mask, cmap="gray"); axes[row, 3].set_title("Augmented Mask")
    for ax in axes[row]:
        ax.axis("off")

plt.suptitle(f"Rotation {cfg.rotation_range}°, Scale {cfg.scale_range} "
             f"(rotate-and-crop: scale clamped to cover canvas, no border fill)", y=1.0)
plt.tight_layout(); plt.show()

In [ ]:
# ============ 14. Fine-tuning helpers + loaders + model (train.py) ============
def dice_score(preds, targets, smooth=1e-6):
    preds = preds.astype(np.float32)
    targets = targets.astype(np.float32)
    intersection = np.sum(preds * targets)
    return (2.0 * intersection + smooth) / (np.sum(preds) + np.sum(targets) + smooth)


def set_bn_eval(m):
    if "BatchNorm" in m.__class__.__name__:
        m.eval()


def evaluate(model, loader, device):
    model.eval()
    running_loss = 0.0
    all_preds_bin, all_masks, dice_scores = [], [], []
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            running_loss += total_loss(preds, masks).item()
            probs = torch.sigmoid(preds)
            pred_bin = (probs > 0.5).float().cpu().numpy()
            mask_np = masks.cpu().numpy()
            dice_scores.append(dice_score(pred_bin, mask_np))
            all_preds_bin.append(pred_bin.astype(np.uint8).flatten())
            all_masks.append(mask_np.astype(np.uint8).flatten())
    all_masks = np.concatenate(all_masks)
    all_preds_bin = np.concatenate(all_preds_bin)
    return (running_loss / len(loader), np.mean(dice_scores),
            jaccard_score(all_masks, all_preds_bin, zero_division=0),
            accuracy_score(all_masks, all_preds_bin),
            precision_score(all_masks, all_preds_bin, zero_division=0),
            recall_score(all_masks, all_preds_bin, zero_division=0),
            f1_score(all_masks, all_preds_bin, zero_division=0))


all_files = list_image_files(cfg.train_img_dir)
train_files, val_files, _ = split_image_list(all_files, test_size=0.1, val_size=0.15, random_state=42)
print(f"Split: {len(train_files)} train / {len(val_files)} val")

train_ds = MoNuSegDataset(cfg.train_img_dir, cfg.train_mask_dir, target_size=cfg.target_size,
                          augment=cfg.augment, augment_factor=cfg.augment_factor,
                          augment_prob=cfg.augment_prob, rotation_range=cfg.rotation_range,
                          scale_range=cfg.scale_range, image_list=train_files)
val_ds = MoNuSegDataset(cfg.train_img_dir, cfg.train_mask_dir, target_size=cfg.target_size,
                        augment=False, image_list=val_files)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=cfg.num_workers, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                        num_workers=cfg.num_workers, pin_memory=True)

model = SegModel(backbone_name=cfg.backbone, pretrained=cfg.pretrained,
                 num_classes=cfg.num_classes).to(device)

if cfg.pretrained_weights and os.path.exists(cfg.pretrained_weights):
    print(f"Loading UNITPathSSL pretrained checkpoint: {cfg.pretrained_weights}")
    checkpoint = torch.load(cfg.pretrained_weights, map_location=device)
    state_dict = checkpoint["state_dict"] if "state_dict" in checkpoint else checkpoint
    model.load_state_dict(state_dict, strict=False)

optimizer = optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs, eta_min=1e-6)

In [ ]:
# ============ 15. Fine-tuning loop (train.py) ============
best_val_dice = 0.0
best_checkpoint = os.path.join(cfg.save_dir, f"{cfg.backbone}_best.pth")
last_checkpoint = os.path.join(cfg.save_dir, f"{cfg.backbone}_last.pth")
log_csv_path = os.path.join(cfg.save_dir, f"{cfg.backbone}_training_metrics.csv")

csv_headers = ["epoch", "train_loss", "val_loss", "val_dice", "val_iou", "val_accuracy",
               "val_precision", "val_recall", "val_f1", "lr", "epoch_time_sec"]
with open(log_csv_path, mode="w", newline="") as f:
    csv.writer(f).writerow(csv_headers)

print(f"Starting training: Backbone={cfg.backbone}, Epochs={cfg.epochs}, "
      f"BatchSize={cfg.batch_size}, LR={cfg.lr}")

epoch_durations = []
total_start_time = time.time()

for epoch in range(1, cfg.epochs + 1):
    epoch_start_time = time.time()
    model.train()
    if cfg.freeze_bn:
        model.apply(set_bn_eval)
    running_loss = 0.0
    optimizer.zero_grad()

    for i, (imgs, masks) in enumerate(train_loader):
        imgs, masks = imgs.to(device), masks.to(device)
        preds = model(imgs)
        loss = total_loss(preds, masks) / cfg.accumulation_steps
        loss.backward()
        if (i + 1) % cfg.accumulation_steps == 0 or (i + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
        running_loss += loss.item() * cfg.accumulation_steps

    current_lr = optimizer.param_groups[0]["lr"]
    scheduler.step()

    train_loss = running_loss / len(train_loader)
    val_loss, val_dice, val_iou, val_acc, val_prec, val_rec, val_f1 = evaluate(model, val_loader, device)

    epoch_elapsed = time.time() - epoch_start_time
    epoch_durations.append(epoch_elapsed)

    with open(log_csv_path, mode="a", newline="") as f:
        csv.writer(f).writerow([
            epoch, f"{train_loss:.6f}", f"{val_loss:.6f}", f"{val_dice:.6f}",
            f"{val_iou:.6f}", f"{val_acc:.6f}", f"{val_prec:.6f}", f"{val_rec:.6f}",
            f"{val_f1:.6f}", f"{current_lr:.8f}", f"{epoch_elapsed:.2f}"])

    print(f"Epoch [{epoch:03d}/{cfg.epochs:03d}] ({epoch_elapsed:.1f}s) | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Val Dice: {val_dice:.4f} | IoU: {val_iou:.4f} | Acc: {val_acc:.4f}")

    torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(), "val_dice": val_dice},
               last_checkpoint)

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        torch.save(model.state_dict(), best_checkpoint)
        print(f"--> Saved Best Model Checkpoint (Val Dice: {best_val_dice:.4f})")

total_elapsed = time.time() - total_start_time
print(f"\n==================== Training Summary ====================")
print(f"Total Training Time:    {total_elapsed/60:.2f} mins")
print(f"Average Time / Epoch:   {sum(epoch_durations)/len(epoch_durations):.2f}s")
print(f"Best Validation Dice:   {best_val_dice:.4f}")
print(f"Best Checkpoint:        {best_checkpoint}")

## Stage D — Held-out Test Evaluation (`test.py`)

In [ ]:
# ============ 16. Test set evaluation (test.py) ============
test_files = list_image_files(cfg.test_img_dir)
test_ds = MoNuSegDataset(cfg.test_img_dir, cfg.test_mask_dir, target_size=cfg.target_size,
                         augment=False, image_list=test_files)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False,
                         num_workers=cfg.num_workers)

test_model = SegModel(backbone_name=cfg.backbone, pretrained=False).to(device)
test_model.load_state_dict(torch.load(best_checkpoint, map_location=device))
test_model.eval()

all_preds, all_masks_t, all_probs, dice_list = [], [], [], []
running_loss = 0.0
print(f"Evaluating {len(test_files)} test images using: {best_checkpoint}")

with torch.no_grad():
    for imgs, masks in test_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        preds = test_model(imgs)
        running_loss += total_loss(preds, masks).item()
        probs = torch.sigmoid(preds)
        pred_bin = (probs > 0.5).float().cpu().numpy()
        mask_np = masks.cpu().numpy()
        dice_list.append(dice_score(pred_bin, mask_np))
        all_probs.append(probs.cpu().numpy().flatten())
        all_preds.append(pred_bin.astype(np.uint8).flatten())
        all_masks_t.append(mask_np.astype(np.uint8).flatten())

all_masks_t = np.concatenate(all_masks_t)
all_preds = np.concatenate(all_preds)
all_probs = np.concatenate(all_probs)

print("\n================== TEST SET EVALUATION RESULTS ==================")
print(f"Mean Dice Score  : {np.mean(dice_list):.4f}")
print(f"Mean IoU (Jaccard): {jaccard_score(all_masks_t, all_preds, zero_division=0):.4f}")
print(f"F1-Score         : {f1_score(all_masks_t, all_preds, zero_division=0):.4f}")
print(f"Precision        : {precision_score(all_masks_t, all_preds, zero_division=0):.4f}")
print(f"Recall           : {recall_score(all_masks_t, all_preds, zero_division=0):.4f}")
print(f"Accuracy         : {accuracy_score(all_masks_t, all_preds):.4f}")
print(f"AUC-ROC          : {roc_auc_score(all_masks_t, all_probs):.4f}")
print(f"Mean Loss        : {running_loss / len(test_loader):.4f}")
print("=================================================================")

## Stage E — Outputs
1. **Training curves** (`plot_curves.py`): 2×2 grid — losses, Dice/IoU, Acc/Prec/Recall, LR schedule.
2. **Qualitative panels** (`generate_output_mask.py`): `[Original | Ground Truth | Prediction]` triptychs saved to disk.
3. **Unlabeled inference** (`infer_unlabeled.py`, optional): predicted mask + green overlay for tiles without labels.
4. **Baseline comparison** (`compare_baseline.py`, optional): overlay val-Dice curves of scratch / ImageNet / UNITPathSSL runs.

In [ ]:
# ============ 17. Training curves (plot_curves.py) ============
df = pd.read_csv(log_csv_path)
epochs_x = df["epoch"]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(epochs_x, df["train_loss"], label="Train Loss", color="#d62728", linewidth=1.8)
axes[0, 0].plot(epochs_x, df["val_loss"], label="Val Loss", color="#8c1111", linestyle="--", linewidth=1.8)
axes[0, 0].set_title("Training vs. Validation Loss"); axes[0, 0].set_xlabel("Epoch"); axes[0, 0].set_ylabel("Loss")
axes[0, 0].legend(loc="upper right"); axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(epochs_x, df["val_dice"], label="Val Dice Score", color="#1f77b4", linewidth=1.8)
axes[0, 1].plot(epochs_x, df["val_iou"], label="Val IoU (Jaccard)", color="#0e3d64", linestyle="--", linewidth=1.8)
axes[0, 1].set_title("Validation Dice & IoU Over Epochs"); axes[0, 1].set_xlabel("Epoch"); axes[0, 1].set_ylabel("Score")
axes[0, 1].legend(loc="lower right"); axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(epochs_x, df["val_accuracy"], label="Accuracy", color="#2ca02c", linewidth=1.8)
axes[1, 0].plot(epochs_x, df["val_precision"], label="Precision", color="#17becf", linestyle="--", linewidth=1.8)
axes[1, 0].plot(epochs_x, df["val_recall"], label="Recall", color="#bcbd22", linestyle=":", linewidth=1.8)
axes[1, 0].set_title("Classification & Segmentation Accuracy"); axes[1, 0].set_xlabel("Epoch"); axes[1, 0].set_ylabel("Score")
axes[1, 0].legend(loc="lower right"); axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(epochs_x, df["lr"], label="Learning Rate", color="#9467bd", linewidth=1.8)
axes[1, 1].set_title("Cosine Annealing Learning Rate Schedule"); axes[1, 1].set_xlabel("Epoch"); axes[1, 1].set_ylabel("LR")
axes[1, 1].set_yscale("log"); axes[1, 1].legend(loc="upper right"); axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
curves_path = os.path.join(cfg.save_dir, "training_curves.png")
plt.savefig(curves_path, dpi=300)
plt.show()
print(f"--> Saved training curve plot to: {curves_path}")

In [ ]:
# ============ 18. Qualitative triple-pane panels (generate_output_mask.py) ============
def find_mask_path(mask_dir, base_name):
    candidates = [os.path.join(mask_dir, f"{base_name}_bin_mask.png"),
                  os.path.join(mask_dir, f"{base_name}.png"),
                  os.path.join(mask_dir, f"{base_name} .png"),   # trailing-space names
                  os.path.join(mask_dir, f"{base_name}_mask.png"),
                  os.path.join(mask_dir, f"{base_name}_bin_mask.tif"),
                  os.path.join(mask_dir, f"{base_name}.tif")]
    for path in candidates:
        if os.path.exists(path):
            return path
    return None


def process_set(model, img_dir, mask_dir, img_list, source_label, device, output_folder,
                target_size=(512, 512), display_size=(512, 512), threshold=0.5,
                title_tag="UNITPath Prediction"):
    """Saves [Original | Ground Truth | Prediction] panels to disk."""
    valid_count = 0
    for img_name in img_list:
        original_bgr = cv2.imread(os.path.join(img_dir, img_name))
        if original_bgr is None:
            continue
        base_name = os.path.splitext(img_name)[0]
        h, w = original_bgr.shape[:2]

        img_input = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2RGB)
        img_input = cv2.resize(img_input, tuple(target_size), interpolation=cv2.INTER_LINEAR)
        img_tensor = (torch.from_numpy(img_input).permute(2, 0, 1).float() / 255.0).unsqueeze(0).to(device)

        mask_path = find_mask_path(mask_dir, base_name)
        if mask_path is not None:
            gt_mask = (cv2.imread(mask_path, 0) > 0).astype(np.uint8) * 255
        else:
            gt_mask = np.zeros((h, w), dtype=np.uint8)

        with torch.no_grad():
            prob = torch.sigmoid(model(img_tensor)).squeeze().cpu().numpy()
        pred_mask = (prob > threshold).astype(np.uint8) * 255
        pred_mask = cv2.resize(pred_mask, (w, h), interpolation=cv2.INTER_NEAREST)

        disp_w, disp_h = display_size
        pane1 = cv2.resize(original_bgr, (disp_w, disp_h))
        pane2 = cv2.resize(cv2.cvtColor(gt_mask, cv2.COLOR_GRAY2BGR), (disp_w, disp_h))
        pane3 = cv2.resize(cv2.cvtColor(pred_mask, cv2.COLOR_GRAY2BGR), (disp_w, disp_h))

        gap = np.zeros((disp_h, 10, 3), dtype=np.uint8)
        combined = np.hstack((pane1, gap, pane2, gap, pane3))

        font, color = cv2.FONT_HERSHEY_SIMPLEX, (0, 255, 0)
        cv2.putText(combined, f"Source: {source_label}", (10, disp_h - 20), font, 0.7, (255, 255, 255), 2)
        cv2.putText(combined, "Original", (10, 35), font, 0.9, color, 2)
        cv2.putText(combined, "Ground Truth", (disp_w + 20, 35), font, 0.9, color, 2)
        cv2.putText(combined, title_tag, (2 * disp_w + 30, 35), font, 0.9, color, 2)

        save_path = os.path.join(output_folder, f"{source_label}_{base_name}.png")
        cv2.imwrite(save_path, combined)
        valid_count += 1
    return valid_count


n = process_set(test_model, cfg.test_img_dir, cfg.test_mask_dir, test_files, "Test",
                device, cfg.panels_dir, target_size=cfg.target_size,
                threshold=cfg.threshold, title_tag=cfg.title_tag)
print(f"Saved {n} panels to {cfg.panels_dir}")

# Display the first 3 panels inline
panel_files = sorted(glob.glob(os.path.join(cfg.panels_dir, "*.png")))[:3]
for pf in panel_files:
    plt.figure(figsize=(16, 6))
    plt.imshow(cv2.cvtColor(cv2.imread(pf), cv2.COLOR_BGR2RGB))
    plt.title(os.path.basename(pf)); plt.axis("off"); plt.show()

In [ ]:
# ============ 19. Inference on unlabeled tiles (infer_unlabeled.py) — optional ============
if cfg.RUN_UNLABELED_INFERENCE:
    os.makedirs(cfg.unlabeled_out_dir, exist_ok=True)
    valid_exts = (".png", ".tif", ".tiff", ".jpg", ".jpeg")
    tissue_files = sorted([f for f in glob.glob(os.path.join(cfg.tissue_dir, "**", "*.*"), recursive=True)
                           if f.lower().endswith(valid_exts)])
    print(f"Found {len(tissue_files)} unlabeled images. Processing first 30...")

    with torch.no_grad():
        for img_path in tissue_files[:30]:
            img = cv2.imread(img_path)
            if img is None:
                continue
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img_resized = cv2.resize(img_rgb, cfg.target_size, interpolation=cv2.INTER_LINEAR)
            img_tensor = (torch.from_numpy(img_resized).permute(2, 0, 1).float() / 255.0).unsqueeze(0).to(device)

            probs = torch.sigmoid(test_model(img_tensor)).squeeze().cpu().numpy()
            pred_bin = (probs > cfg.threshold).astype(np.uint8) * 255

            base_name = os.path.splitext(os.path.basename(img_path))[0]
            cv2.imwrite(os.path.join(cfg.unlabeled_out_dir, f"{base_name}_pred_mask.png"), pred_bin)

            overlay = img_resized.copy()
            overlay[pred_bin > 0] = [0, 255, 0]
            blended = cv2.addWeighted(img_resized, 0.6, overlay, 0.4, 0)
            comparison = np.hstack([cv2.cvtColor(img_resized, cv2.COLOR_RGB2BGR),
                                    cv2.cvtColor(pred_bin, cv2.COLOR_GRAY2BGR),
                                    cv2.cvtColor(blended, cv2.COLOR_RGB2BGR)])
            cv2.imwrite(os.path.join(cfg.unlabeled_out_dir, f"{base_name}_comparison.png"), comparison)
    print(f"Generated masks saved in: {cfg.unlabeled_out_dir}")
else:
    print("Unlabeled inference skipped (RUN_UNLABELED_INFERENCE=False).")

In [ ]:
# ============ 20. Baseline comparison (compare_baseline.py) — optional ============
# Overlays val-Dice convergence of multiple runs. Each run writes its own metrics CSV;
# re-run Stage C with different init (pretrained=False / ImageNet / UNITPathSSL ckpt)
# and different save_dir, then list the CSVs here.
baseline_runs = {
    # "Scratch (Random)":    "/kaggle/working/checkpoints/ablation_scratch/resnet50_training_metrics.csv",
    # "ImageNet Pretrained": "/kaggle/working/checkpoints/ablation_imagenet/resnet50_training_metrics.csv",
    "Current Run": log_csv_path,
}

plt.figure(figsize=(10, 6))
styles = [("gray", ":"), ("blue", "--"), ("green", "-")]
for (label, csv_file), (color, ls) in zip(baseline_runs.items(), styles):
    if os.path.exists(csv_file):
        d = pd.read_csv(csv_file)
        plt.plot(d["epoch"], d["val_dice"], label=label, color=color, linestyle=ls,
                 linewidth=2.5 if ls == "-" else 1.8)
plt.title("Validation Dice Score Convergence", fontsize=14)
plt.xlabel("Epoch"); plt.ylabel("Validation Dice")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig("/kaggle/working/checkpoints/pretrain_advancement_comparison.png", dpi=300)
plt.show()

## Outputs & Notes
- **Checkpoints + metrics CSV:** `/kaggle/working/checkpoints/final/`
- **Qualitative panels:** `/kaggle/working/output/test_panels/`
- All downloadable from the notebook **Output** tab after saving a version.

**Not included:** `RCNN.py` (auxiliary Mask R-CNN used as the Instance-Segmentation-Guided network — a separate experiment, torchvision-only so it can be added if needed), `grid_search.py` (KFold hyperparameter sweep — too long for one Kaggle session; the thesis hyperparameters in `Config` are its result).